Importing the Dependencies

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn import svm
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import joblib

Data Collection and Analysis

Diabetes Dataset

In [2]:
diabetes_dataset = pd.read_csv('../Dataset/diabetes.csv')

In [ ]:
# Convert invalid zero values to NaN (domain knowledge: these columns cannot have zero values)
cols_with_invalid_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in cols_with_invalid_zeros:
    diabetes_dataset[col] = diabetes_dataset[col].replace(0, np.nan)

print("Missing values after zero-to-NaN conversion:")
print(diabetes_dataset[cols_with_invalid_zeros].isnull().sum())

In [7]:
diabetes_dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [9]:
X = diabetes_dataset.drop(columns = 'Outcome')
Y = diabetes_dataset['Outcome']

In [11]:
print(Y)

0      1
1      0
2      1
3      0
4      1
      ..
763    0
764    0
765    0
766    1
767    0
Name: Outcome, Length: 768, dtype: int64


In [12]:
X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size = 0.2, stratify=Y, random_state=2)

In [ ]:
# Create Pipeline: Imputer -> Scaler -> SVC
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', svm.SVC(kernel='linear'))
])

print("Pipeline created:")
print(pipeline)

In [ ]:
# Fit Pipeline on training data only (imputer and scaler learn from X_train only)
pipeline.fit(X_train, Y_train)
print("Pipeline fitted on training data")

In [ ]:
# Model Evaluation using Pipeline (handles imputation + scaling + prediction internally)
X_train_prediction = pipeline.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction, Y_train)

print(f'Accuracy score of the training data : {training_data_accuracy*100:.2f}%')

Accuracy Score

In [ ]:
# Comprehensive Model Evaluation on Test Data
X_test_prediction = pipeline.predict(X_test)
test_data_accuracy = accuracy_score(X_test_prediction, Y_test)

print(f'Accuracy score of the test data : {test_data_accuracy*100:.2f}%')

# Confusion Matrix
cm = confusion_matrix(Y_test, X_test_prediction)
print(f'\nConfusion Matrix:')
print(cm)
print(f'  True Negatives: {cm[0,0]}, False Positives: {cm[0,1]}')
print(f'  False Negatives: {cm[1,0]}, True Positives: {cm[1,1]}')

# Classification Report
print('\nClassification Report:')
print(classification_report(Y_test, X_test_prediction, target_names=['Non-Diabetic', 'Diabetic']))

# Individual Metrics
precision = precision_score(Y_test, X_test_prediction)
recall = recall_score(Y_test, X_test_prediction)
f1 = f1_score(Y_test, X_test_prediction)

print(f'Precision: {precision:.3f}')
print(f'Recall (Sensitivity): {recall:.3f}')
print(f'F1-Score: {f1:.3f}')

# ROC-AUC using decision_function
try:
    y_scores = pipeline.decision_function(X_test)
    roc_auc = roc_auc_score(Y_test, y_scores)
    print(f'ROC-AUC: {roc_auc:.3f}')
except Exception as e:
    print(f'ROC-AUC could not be computed: {e}')

In [ ]:
# Making a Predictive System using the Pipeline
input_data = (5,166,72,19,175,25.8,0.587,51)

columns = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
           'Insulin','BMI','DiabetesPedigreeFunction','Age']

input_df = pd.DataFrame([input_data], columns=columns)

prediction = pipeline.predict(input_df)

print(prediction)

if prediction[0] == 0:
    print('The person is not diabetic')
else:
    print('The person is diabetic')

In [ ]:
# Saving the complete Pipeline (imputer + scaler + classifier)
filename = '../saved_models/diabetes_pipeline.joblib'
joblib.dump(pipeline, filename)
print(f'Pipeline saved to {filename}')

In [25]:
for column in X.columns:
  print(column)

Pregnancies
Glucose
BloodPressure
SkinThickness
Insulin
BMI
DiabetesPedigreeFunction
Age
